In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import rasterio
from rasterio.enums import Resampling

In [ ]:
for i in range(0,1440,60):
    print(i)
    if i == 0:
        data = pd.read_csv(f"hydrogen_production_results_{i}.csv")
        data = np.array(data)
        for lat in range(0,721):
            for lon in range(0,60):
                data[lat*60+lon,0]=(lat*0.25-90)*-1
                data[lat*60+lon,1]=(lon+i)*0.25
        # 提取纬度和经度
        latitude = np.unique(data[:, 0])  # 纬度
        longitude = np.unique(data[:, 1])  # 经度

        # 提取第3到第7列的变量
        variables = {
            "wind_capacity": data[:, 2].astype(float),
            "pv_capacity": data[:, 3].astype(float),
            "battery_capacity": data[:, 4].astype(float),
            "electrolyzer_capacity": data[:, 5].astype(float),
            "hydrogen_cost": data[:, 6].astype(float)
        }
        # 构建 xarray.Dataset
        dataset = xr.Dataset(
                {
                    "wind_capacity": (["latitude", "longitude"], data[:, 2].astype(float).reshape(721, 60)),
                    "pv_capacity": (["latitude", "longitude"], data[:, 3].astype(float).reshape(721, 60)),
                    "battery_capacity": (["latitude", "longitude"], data[:, 4].astype(float).reshape(721, 60)),
                    "electrolyzer_capacity": (["latitude", "longitude"], data[:, 5].astype(float).reshape(721, 60)),
                    "hydrogen_cost": (["latitude", "longitude"], data[:, 6].astype(float).reshape(721, 60)),
                },
                coords={
                    "latitude": latitude,
                    "longitude":longitude
                }
            )

    else: 
        data0 = pd.read_csv(f"hydrogen_production_results_{i}.csv")
        print(f"hydrogen_production_results_{i}.csv")
        data0 = np.array(data0)
        for lat in range(0,721):
            for lon in range(0,60):
                data0[lat*60+lon,0]=(lat*0.25-90)*-1
                data0[lat*60+lon,1]=(lon+i)*0.25
                
        # 提取纬度和经度
        latitude = np.unique(data0[:, 0])  # 纬度
        longitude = np.unique(data0[:, 1])  # 经度

        # 提取第3到第7列的变量
        variables = {
            "wind_capacity": data0[:, 2].astype(float),
            "pv_capacity": data0[:, 3].astype(float),
            "battery_capacity": data0[:, 4].astype(float),
            "electrolyzer_capacity": data0[:, 5].astype(float),
            "hydrogen_cost": data0[:, 6].astype(float)
        }
        # 构建 xarray.Dataset
        dataset0 = xr.Dataset(
                {
                    "wind_capacity": (["latitude", "longitude"], data0[:, 2].astype(float).reshape(721, 60)),
                    "pv_capacity": (["latitude", "longitude"], data0[:, 3].astype(float).reshape(721, 60)),
                    "battery_capacity": (["latitude", "longitude"], data0[:, 4].astype(float).reshape(721, 60)),
                    "electrolyzer_capacity": (["latitude", "longitude"], data0[:, 5].astype(float).reshape(721, 60)),
                    "hydrogen_cost": (["latitude", "longitude"], data0[:, 6].astype(float).reshape(721, 60))
                },
                coords={
                    "latitude": latitude,
                    "longitude":longitude
                }
            )        
        dataset = xr.concat([dataset, dataset0], dim="longitude")

In [ ]:
# 提取 hydrogen_cost 变量
hydrogen_cost = dataset["hydrogen_cost"]

# 掩盖 0 值
masked_data = np.ma.masked_where(hydrogen_cost <= 0, hydrogen_cost)

# 创建颜色映射
cmap = plt.cm.viridis.reversed()
cmap.set_under('white')  # 为 0 值指定颜色

# 绘制图像
plt.imshow(masked_data, cmap=cmap, vmin=0.001)  # 设置 vmin > 0 避免显示默认颜色
plt.colorbar()
plt.title("Special Color for Zero Value in Hydrogen Cost")
plt.show()

In [ ]:
masked_data

In [ ]:
plt.imshow(dataset["hydrogen_cost"])
plt.colorbar()

In [ ]:
plt.imshow(masked_data[148:300,292:530])
plt.colorbar()